In [1]:
import pandas as pd
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix, f1_score
from sklearn.exceptions import ConvergenceWarning
import warnings

warnings.filterwarnings("ignore", category=ConvergenceWarning)

In [2]:
# Constants
TRAIN_SIZE = 0.2
RANDOM_STATE = 1082

real_data_path = "../../../Datasets/excel/2dSpoofingDataset/GPS_Data_Simplified_2D_Feature_Map.xlsx"
base_dir = "../../../Datasets/excel/2dSpoofingDataset/"
RESULTS_CSV = "../../../2D_Results.csv"

models_list = [
    "ctgan",
    "gc",
    "tvae",
]
sample_sizes = [10_000, 100_000, 1_000_000]

In [3]:
# Load synthetic datasets
synthetic_dfs = {}
for model in models_list:
    for n in sample_sizes:
        file_name = f"spoofing_{model.lower()}_{n}.csv"
        file_path = os.path.join(base_dir, file_name)
        try:
            synthetic_dfs[file_name] = pd.read_csv(file_path)
            print(f"Loaded {file_name}")
        except Exception as e:
            print(f"Could not load {file_name}: {e}")

# Load real data
try:
    real_df = pd.read_excel(real_data_path)
    print("Loaded real data")
except Exception as e:
    print(f"Could not load real data: {e}")
    real_df = None

Loaded spoofing_ctgan_10000.csv
Loaded spoofing_ctgan_100000.csv
Loaded spoofing_ctgan_1000000.csv
Loaded spoofing_gc_10000.csv
Loaded spoofing_gc_100000.csv
Loaded spoofing_gc_1000000.csv
Loaded spoofing_tvae_10000.csv
Loaded spoofing_tvae_100000.csv
Loaded spoofing_tvae_1000000.csv
Loaded real data


In [4]:
# Preprocess real data
if real_df is not None:
    if 'Output' not in real_df.columns:
        raise ValueError("Real data does not have 'Output' column.")
    X_real = real_df.drop('Output', axis=1)
    y_real = real_df['Output']
    if y_real.dtype == object or y_real.dtype.name == 'category':
        le_real = LabelEncoder()
        y_real = le_real.fit_transform(y_real)
else:
    raise RuntimeError("Real data could not be loaded.")

In [5]:
# Models to evaluate
models = {
    "Logistic Regression": LogisticRegression(random_state=RANDOM_STATE, max_iter=5000, solver='saga'),
    "MLP Classifier": MLPClassifier(hidden_layer_sizes=(100, 50, 25), activation='tanh', solver='adam',
                                    random_state=RANDOM_STATE, learning_rate='adaptive', early_stopping=True, ),
    "XGBoost": XGBClassifier(
        objective='multi:softprob',
        num_class=4,
        eval_metric='mae',
        random_state=RANDOM_STATE,
    )
}

In [6]:
results = []
for synth_name, synth_df in synthetic_dfs.items():
    print(f"\nTrain on synthetic: {synth_name}, testing on real data")

    if 'Output' not in synth_df.columns:
        print(f"Skipping {synth_name}: 'Output' column not found.")
        continue

    X_synth = synth_df.drop('Output', axis=1)
    y_synth = synth_df['Output']
    if y_synth.dtype == object or y_synth.dtype.name == 'category':
        le_synth = LabelEncoder()
        y_synth = le_synth.fit_transform(y_synth)

    X_synth_train, _, y_synth_train, _ = train_test_split(
        X_synth, y_synth, train_size=TRAIN_SIZE, random_state=RANDOM_STATE, stratify=y_synth
    )

    scaler = StandardScaler()
    X_synth_train_scaled = scaler.fit_transform(X_synth_train)
    X_real_scaled = scaler.transform(X_real)

    for model_name, model in models.items():
        print(f"\n{model_name}")
        print("Training on synthetic")
        model.fit(X_synth_train_scaled, y_synth_train)
        print("Evaluating on real data...")
        y_pred = model.predict(X_real_scaled)
        acc = accuracy_score(y_real, y_pred)
        f1 = f1_score(y_real, y_pred, average='weighted', zero_division=0)
        print(f"Accuracy: {acc:.4f}")
        print(f"F1 Score: {f1:.4f}")
        print("\nClassification Report:")
        print(classification_report(y_real, y_pred, zero_division=0))
        print("Confusion Matrix:")
        print(confusion_matrix(y_real, y_pred))

        results.append({
            "model": model_name,
            "synthetic_dataset": synth_name,
            "f1score": f1,
            "accuracy": acc
        })


Train on synthetic: spoofing_ctgan_10000.csv, testing on real data

Logistic Regression
Training on synthetic
Evaluating on real data...
Accuracy: 0.4384
F1 Score: 0.5089

Classification Report:
              precision    recall  f1-score   support

           0       0.86      0.45      0.59    397825
           1       0.17      0.37      0.23     36458
           2       0.19      0.31      0.23     44232
           3       0.12      0.54      0.19     32015

    accuracy                           0.44    510530
   macro avg       0.33      0.42      0.31    510530
weighted avg       0.70      0.44      0.51    510530

Confusion Matrix:
[[179074  57875  49332 111544]
 [ 10235  13662   5710   6851]
 [ 11707   8163  13743  10619]
 [  7915   2492   4282  17326]]

MLP Classifier
Training on synthetic
Evaluating on real data...
Accuracy: 0.4522
F1 Score: 0.5042

Classification Report:
              precision    recall  f1-score   support

           0       0.95      0.39      0.55    3

In [7]:
# add results to CSV
results_df = pd.DataFrame(results)
file_exists = os.path.isfile(RESULTS_CSV)
results_df.to_csv(
    RESULTS_CSV,
    mode='a',
    header=not file_exists,
    index=False
)
print(f"Results appended to {RESULTS_CSV}")

Results appended to ../../../2D_Results.csv
